# Multi-head attention

_Modern Deep Learning & AI_

**Run several attention heads at once. Each learns a different relationship, then you stitch them together.**

One attention pattern can only focus on one kind of relationship at a time.

     Multi-head attention runs several attentions side by side. Each head has its own query, key, and value vectors, so each learns to focus on something different.

---

This notebook is a practice scaffold for the **Multi-head attention** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
batch, seq, d_model, heads = 1, 4, 16, 4          # 16 dims split into 4 heads of 4
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, batch_first=True)

x = torch.randn(batch, seq, d_model)              # tiny synthetic input
# self-attention: query, key, value all come from x
out, weights = mha(x, x, x, need_weights=True)
print("output shape:", out.shape)                 # (1, 4, 16)
print("attn weights shape:", weights.shape)       # (1, 4, 4) averaged over heads
print("each query row sums to:", weights[0, 0].sum().item())  # ~1.0

# each head works in d_model // heads dims; concat returns to d_model
print("per-head dim:", d_model // heads)           # 4

## Visualize the data & results

_With 4 heads on the real sentence, how do different heads route the word it?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# multi-head attention on the REAL sentence: per-head softmax, then average
tokens = ["The","animal","didnt","cross","the","street",
          "because","it","was","too","tired"]
seq, d_model, heads = len(tokens), 16, 4
d_head = d_model // heads
rng = np.random.default_rng(7)
X = np.stack([np.random.default_rng(1000+i).standard_normal(d_model)
              for i in range(seq)])
X[7] = 0.30*X[7] + 0.95*X[1]           # "it" ~ "animal"

def attn(q, k):
    s = q @ k.T / np.sqrt(q.shape[-1])
    s -= s.max(axis=1, keepdims=True)
    w = np.exp(s); return w / w.sum(axis=1, keepdims=True)

maps = []
for h in range(heads):
    Wq = rng.standard_normal((d_model, d_head)) / np.sqrt(d_head)
    Wk = rng.standard_normal((d_model, d_head)) / np.sqrt(d_head)
    maps.append(attn(X @ Wq, X @ Wk))
maps = np.stack(maps)                   # (heads, seq, seq)
avg = maps.mean(axis=0)
it = 7                                  # query row for "it"

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
im = axes[0].imshow(avg, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("averaged over 4 heads")
fig.colorbar(im, ax=axes[0])
axes[1].bar(tokens, maps[0, it], color="#4ea1ff"); axes[1].set_title("it: head 1")
axes[2].bar(tokens, maps[1, it], color="#7ee787"); axes[2].set_title("it: head 2")
for a in axes[1:]: a.set_xticklabels(tokens, rotation=90)
plt.show()